# Governance & Security Tests

## Test Cases
| Capability | Test Case | Target |
|------------|-----------|--------|
| Dynamic Data Masking | Apply masking to SSN/email, test by role | Verify CLONE/CTAS behavior |
| Row Access Policy | Create region mapping, apply RAP | 100% filtering accuracy |
| Tag Propagation | Apply tags at schema/table/column | Verify in TAG_REFERENCES |
| Access History | Execute all DML operations | 100% audit coverage |
| Tokenization | Configure format-preserving encryption | Test de-tokenization by role |

In [ ]:
-- Set context
USE ROLE ACCOUNTADMIN;
USE DATABASE ICEBERG_POC;
USE SCHEMA TESTS;
USE WAREHOUSE COMPUTE_WH;

## Test 1: Dynamic Data Masking
Apply masking policy to PII columns and verify enforcement.

In [ ]:
-- Create test table with PII
CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.TESTS.CUSTOMERS_PII (
    id INT,
    name STRING,
    email STRING,
    ssn STRING,
    phone STRING,
    region STRING
)
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'EXVOL'
ICEBERG_VERSION = 3;

-- Insert sample PII data
INSERT INTO ICEBERG_POC.TESTS.CUSTOMERS_PII
SELECT 
    SEQ4() AS id,
    'Customer_' || SEQ4() AS name,
    'customer' || SEQ4() || '@company.com' AS email,
    LPAD(UNIFORM(100000000, 999999999, RANDOM())::STRING, 9, '0') AS ssn,
    '555-' || LPAD(UNIFORM(1000, 9999, RANDOM())::STRING, 4, '0') AS phone,
    ARRAY_CONSTRUCT('US-EAST', 'US-WEST', 'EU', 'APAC')[UNIFORM(0, 3, RANDOM())] AS region
FROM TABLE(GENERATOR(ROWCOUNT => 10000));

In [ ]:
-- Create masking policies
CREATE OR REPLACE MASKING POLICY ICEBERG_POC.TESTS.MASK_SSN AS (val STRING) RETURNS STRING ->
    CASE 
        WHEN IS_ROLE_IN_SESSION('ACCOUNTADMIN') THEN val
        ELSE '***-**-' || RIGHT(val, 4)
    END;

CREATE OR REPLACE MASKING POLICY ICEBERG_POC.TESTS.MASK_EMAIL AS (val STRING) RETURNS STRING ->
    CASE 
        WHEN IS_ROLE_IN_SESSION('ACCOUNTADMIN') THEN val
        ELSE REGEXP_REPLACE(val, '(.)[^@]*(@.*)', '\\1****\\2')
    END;

-- Apply masking policies to Iceberg table
ALTER ICEBERG TABLE ICEBERG_POC.TESTS.CUSTOMERS_PII MODIFY COLUMN ssn SET MASKING POLICY ICEBERG_POC.TESTS.MASK_SSN;
ALTER ICEBERG TABLE ICEBERG_POC.TESTS.CUSTOMERS_PII MODIFY COLUMN email SET MASKING POLICY ICEBERG_POC.TESTS.MASK_EMAIL;

In [ ]:
-- Test masking enforcement (as ACCOUNTADMIN - should see full data)
SELECT id, name, email, ssn FROM ICEBERG_POC.TESTS.CUSTOMERS_PII LIMIT 5;

-- Test CLONE/CTAS behavior with masking
CREATE OR REPLACE TABLE ICEBERG_POC.TESTS.CUSTOMERS_PII_COPY AS 
SELECT * FROM ICEBERG_POC.TESTS.CUSTOMERS_PII LIMIT 100;

SELECT * FROM ICEBERG_POC.TESTS.CUSTOMERS_PII_COPY LIMIT 5;

## Test 2: Row Access Policy
Create region-based filtering and verify 100% accuracy.

In [ ]:
-- Create region mapping table
CREATE OR REPLACE TABLE ICEBERG_POC.TESTS.REGION_ACCESS (
    role_name STRING,
    allowed_region STRING
);

INSERT INTO ICEBERG_POC.TESTS.REGION_ACCESS VALUES
    ('ANALYST_US', 'US-EAST'),
    ('ANALYST_US', 'US-WEST'),
    ('ANALYST_EU', 'EU'),
    ('ANALYST_APAC', 'APAC'),
    ('ACCOUNTADMIN', 'US-EAST'),
    ('ACCOUNTADMIN', 'US-WEST'),
    ('ACCOUNTADMIN', 'EU'),
    ('ACCOUNTADMIN', 'APAC');

-- Create Row Access Policy
CREATE OR REPLACE ROW ACCESS POLICY ICEBERG_POC.TESTS.REGION_RAP AS (region STRING) RETURNS BOOLEAN ->
    EXISTS (
        SELECT 1 FROM ICEBERG_POC.TESTS.REGION_ACCESS 
        WHERE role_name = CURRENT_ROLE() AND allowed_region = region
    );

-- Apply RAP to Iceberg table
ALTER ICEBERG TABLE ICEBERG_POC.TESTS.CUSTOMERS_PII ADD ROW ACCESS POLICY ICEBERG_POC.TESTS.REGION_RAP ON (region);

## Test 3: Tag Propagation
Apply tags at schema/table/column levels and verify in TAG_REFERENCES.

In [ ]:
-- Create tags
CREATE OR REPLACE TAG ICEBERG_POC.TESTS.PII_TYPE ALLOWED_VALUES 'SSN', 'EMAIL', 'PHONE', 'NAME';
CREATE OR REPLACE TAG ICEBERG_POC.TESTS.DATA_CLASSIFICATION ALLOWED_VALUES 'CONFIDENTIAL', 'INTERNAL', 'PUBLIC';
CREATE OR REPLACE TAG ICEBERG_POC.TESTS.COST_CENTER;

-- Apply tags to Iceberg table and columns
ALTER ICEBERG TABLE ICEBERG_POC.TESTS.CUSTOMERS_PII SET TAG ICEBERG_POC.TESTS.DATA_CLASSIFICATION = 'CONFIDENTIAL';
ALTER ICEBERG TABLE ICEBERG_POC.TESTS.CUSTOMERS_PII MODIFY COLUMN ssn SET TAG ICEBERG_POC.TESTS.PII_TYPE = 'SSN';
ALTER ICEBERG TABLE ICEBERG_POC.TESTS.CUSTOMERS_PII MODIFY COLUMN email SET TAG ICEBERG_POC.TESTS.PII_TYPE = 'EMAIL';
ALTER ICEBERG TABLE ICEBERG_POC.TESTS.CUSTOMERS_PII MODIFY COLUMN phone SET TAG ICEBERG_POC.TESTS.PII_TYPE = 'PHONE';

-- Verify tags
SELECT * FROM TABLE(INFORMATION_SCHEMA.TAG_REFERENCES('ICEBERG_POC.TESTS.CUSTOMERS_PII', 'TABLE'));
SELECT * FROM TABLE(INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS('ICEBERG_POC.TESTS.CUSTOMERS_PII', 'TABLE'));

## Test 4: Access History Audit
Verify 100% audit coverage for all operations in ACCESS_HISTORY.

In [ ]:
-- Execute various operations to generate audit trail
SELECT COUNT(*) FROM ICEBERG_POC.TESTS.CUSTOMERS_PII;
INSERT INTO ICEBERG_POC.TESTS.CUSTOMERS_PII VALUES (99999, 'Audit_Test', 'audit@test.com', '000000000', '555-0000', 'US-EAST');
UPDATE ICEBERG_POC.TESTS.CUSTOMERS_PII SET name = 'Audit_Updated' WHERE id = 99999;
DELETE FROM ICEBERG_POC.TESTS.CUSTOMERS_PII WHERE id = 99999;

-- Query ACCESS_HISTORY for Iceberg table operations (may take a few minutes to populate)
SELECT 
    query_start_time,
    user_name,
    query_type,
    direct_objects_accessed
FROM SNOWFLAKE.ACCOUNT_USAGE.ACCESS_HISTORY
WHERE ARRAY_CONTAINS('ICEBERG_POC.TESTS.CUSTOMERS_PII'::VARIANT, 
    TRANSFORM(direct_objects_accessed, o -> o:objectName::STRING))
ORDER BY query_start_time DESC
LIMIT 20;